# 01 · Data Cleaning & Feature Engineering

**Goal:** turn the raw IBM Telco extract into a tidy, analysis-ready table that every later notebook can trust.

Steps:
1. Load and inspect structure and data types
2. Fix `TotalCharges` (stored as text, with blanks)
3. Standardise labels (`SeniorCitizen`, `Churn`) and add a 0/1 churn flag
4. Check duplicates and inconsistent category labels
5. Engineer features: `tenure_band`, `num_services`, `has_streaming`, `avg_monthly_spend`
6. Save to `data/processed/clean.csv`

The shape is reported after every step so no rows silently disappear.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.churn_utils import RAW_PATH, CLEAN_PATH, TENURE_BANDS

pd.set_option("display.max_columns", 30)

def report(df, step):
    print(f"[{step}] shape = {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Load and inspect

In [2]:
raw = pd.read_csv(RAW_PATH)
report(raw, "loaded raw")
raw.head()

[loaded raw] shape = 7,043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
raw.dtypes.to_frame("dtype").T

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
dtype,object,object,int64,object,object,int64,object,object,object,object,object,object,object,object,object,object,object,object,float64,object,object


In [4]:
print("Missing values (NaN) in raw file:", raw.isna().sum().sum())
raw.describe().round(2)

Missing values (NaN) in raw file: 0


,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


`SeniorCitizen` is stored as 0/1 while every other yes/no field is text, and `TotalCharges` is stored as **object (text)** even though it is money. Both need fixing.

## 2. `TotalCharges`: text to numeric

In [5]:
df = raw.copy()
blank_mask = df["TotalCharges"].str.strip() == ""
print("Blank TotalCharges strings:", blank_mask.sum())
df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Blank TotalCharges strings: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("NaN after conversion:", df["TotalCharges"].isna().sum())
print("Tenure of those rows:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
print("Customers with tenure 0 in total:", (df["tenure"] == 0).sum())

NaN after conversion: 11
Tenure of those rows: [0]
Customers with tenure 0 in total: 11


**Decision: impute `TotalCharges = 0` for these rows, and keep them.**

- Every blank belongs to a customer with **tenure = 0**, and every tenure-0 customer has a blank. These are brand-new sign-ups who have not been billed yet. Their true lifetime charges are 0, so the value is not really missing.
- Dropping them would remove real, active customers from the base and slightly overstate the churn rate. None of them have churned.
- Imputing the median or `MonthlyCharges` would invent revenue that was never billed.

In [7]:
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
report(df, "TotalCharges numeric")
df["TotalCharges"].dtype, df["TotalCharges"].isna().sum()

[TotalCharges numeric] shape = 7,043 rows x 21 columns


(dtype('float64'), np.int64(0))

## 3. Consistent labels and churn flag

In [8]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["Churn"] = df["Churn"].str.strip().str.title()
df["churn_flag"] = (df["Churn"] == "Yes").astype(int)
report(df, "labels standardised")
pd.crosstab(df["Churn"], df["churn_flag"])

[labels standardised] shape = 7,043 rows x 22 columns


churn_flag,0,1
Churn,,
No,5174,0
Yes,0,1869


## 4. Duplicates and category label check

In [9]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated customerIDs:", df["customerID"].duplicated().sum())

cat_cols = [c for c in df.select_dtypes("object").columns if c != "customerID"]
label_check = pd.DataFrame({
    "levels": [df[c].nunique() for c in cat_cols],
    "values": [sorted(df[c].unique()) for c in cat_cols],
    "whitespace/case issues": [
        (df[c] != df[c].str.strip()).sum() + (df[c].str.strip().str.lower().nunique() != df[c].nunique())
        for c in cat_cols
    ],
}, index=cat_cols)
label_check

Fully duplicated rows: 0
Duplicated customerIDs: 0


,levels,values,whitespace/case issues
gender,2,"[Female, Male]",0
SeniorCitizen,2,"[No, Yes]",0
Partner,2,"[No, Yes]",0
Dependents,2,"[No, Yes]",0
PhoneService,2,"[No, Yes]",0
MultipleLines,3,"[No, No phone service, Yes]",0
InternetService,3,"[DSL, Fiber optic, No]",0
OnlineSecurity,3,"[No, No internet service, Yes]",0
OnlineBackup,3,"[No, No internet service, Yes]",0
DeviceProtection,3,"[No, No internet service, Yes]",0


No duplicates and no stray whitespace or case variants. The only label quirk is that `"No internet service"` and `"No phone service"` are separate levels for add-on columns. **They are kept as-is** because they describe a different customer (one with no internet or phone line) from a customer who chose not to buy the add-on. Collapsing them into "No" would hide that difference in the driver analysis.

In [10]:
report(df, "after duplicate / label checks")

[after duplicate / label checks] shape = 7,043 rows x 22 columns


## 5. Feature engineering

| Feature | Definition |
|---|---|
| `tenure_band` | 0-12, 13-24, 25-48, 49-72 months |
| `num_services` | Count of services held: phone, multiple lines, internet, and the six internet add-ons (0-9) |
| `has_streaming` | Yes if the customer streams TV **or** movies |
| `avg_monthly_spend` | `TotalCharges / tenure`; for tenure-0 customers this falls back to `MonthlyCharges` (no division by zero) |

In [11]:
df["tenure_band"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72], labels=TENURE_BANDS)

addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["num_services"] = (
    (df["PhoneService"] == "Yes").astype(int)
    + (df["MultipleLines"] == "Yes").astype(int)
    + (df["InternetService"] != "No").astype(int)
    + (df[addon_cols] == "Yes").sum(axis=1)
)
df["has_streaming"] = np.where((df["StreamingTV"] == "Yes") | (df["StreamingMovies"] == "Yes"), "Yes", "No")
df["avg_monthly_spend"] = np.where(df["tenure"] > 0, df["TotalCharges"] / df["tenure"].replace(0, np.nan), df["MonthlyCharges"]).round(2)
report(df, "features engineered")

[features engineered] shape = 7,043 rows x 26 columns


In [12]:
df["tenure_band"].value_counts().sort_index().to_frame("customers")

,customers
tenure_band,
0-12,2186
13-24,1024
25-48,1594
49-72,2239


In [13]:
df[["num_services", "avg_monthly_spend"]].describe().round(2)

,num_services,avg_monthly_spend
count,7043.00,7043.00
mean,4.15,64.76
std,2.31,30.19
min,1.00,13.78
25%,2.00,35.94
50%,4.00,70.34
75%,6.00,90.17
max,9.00,121.40


In [14]:
# Sanity check: average spend over the lifetime should track the current monthly bill
print("Correlation avg_monthly_spend vs MonthlyCharges:",
      round(df["avg_monthly_spend"].corr(df["MonthlyCharges"]), 3))
df.groupby("has_streaming")["churn_flag"].agg(["size", "mean"]).round(3)

Correlation avg_monthly_spend vs MonthlyCharges: 0.996


,size,mean
has_streaming,,
No,3544,0.228
Yes,3499,0.303


## 6. Save

In [15]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
report(df, "saved")
print("Saved to", CLEAN_PATH.relative_to(CLEAN_PATH.parents[2]))

[saved] shape = 7,043 rows x 26 columns
Saved to data\processed\clean.csv


In [16]:
print(f"Rows in: {raw.shape[0]:,}  |  Rows out: {df.shape[0]:,}  |  Rows dropped: {raw.shape[0] - df.shape[0]}")
print(f"Columns in: {raw.shape[1]}  |  Columns out: {df.shape[1]}  (+{df.shape[1] - raw.shape[1]} engineered)")
print(f"Churners: {df['churn_flag'].sum():,} of {len(df):,}")

Rows in: 7,043  |  Rows out: 7,043  |  Rows dropped: 0
Columns in: 21  |  Columns out: 26  (+5 engineered)
Churners: 1,869 of 7,043


## Key Takeaways

- **No rows were lost:** 7,043 customers in, 7,043 out. There are no duplicate rows and no duplicate `customerID`s.
- **`TotalCharges` had 11 blank strings**, all from customers with **tenure = 0** (new sign-ups, none churned). They were set to **0** rather than dropped, because nothing has been billed yet.
- **Labels are now consistent:** `SeniorCitizen` is Yes/No like every other flag, and `churn_flag` (0/1) matches `Churn` exactly, with **1,869 churners and 5,174 retained**.
- **Category labels were already clean:** there were no whitespace or case variants. "No internet service" and "No phone service" were kept on purpose, because they mark a different kind of customer.
- **Five features were added** (21 → 26 columns): `churn_flag`, `tenure_band`, `num_services` (1-9, median 4), `has_streaming` (3,499 customers stream) and `avg_monthly_spend`. `avg_monthly_spend` correlates **0.996** with `MonthlyCharges`, so current bills are a reliable stand-in for lifetime spend.
- **New customers are the biggest group:** 2,186 customers (31%) are in their first 12 months, and the clean file is saved at `data/processed/clean.csv`.